# City Explorer — Tel Aviv Flickr Analysis

End-to-end notebook for the whole project: **Part 1 (supervised)** and **Part 2 (unsupervised + agent simulation)**.

---

## 1. Data Cleaning

The entire cleaning pipeline runs in a single pass below. Steps applied, in order:

1. **Drop nulls** — remove any row with a missing value
2. **Date validity** — keep `2004 ≤ year < 2026` (Flickr launched 2004), valid month/day, and a parseable calendar date (drops e.g. Feb 30)
3. **Bounding box** — keep only coordinates inside the Tel Aviv area
4. **Land check** — drop points that fall in the sea (Natural Earth land polygons)
5. **Duplicate photos** — same user + same photo URL
6. **GPS jitter** — same user, same spot, same timestamp (collapses burst uploads)
7. **URL validation** — keep only well-formed Flickr photo URLs

Output: `flickr_clean.csv`

In [1]:
import re
import pandas as pd
import geopandas as gpd
import geodatasets

/home/roneng/city-explorer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ====================================================================
# DATA CLEANING — single pass: load -> validate -> spatial filter
#                 -> dedup -> save
# ====================================================================
RAW_FILE = "flickr_output_TelAviv100.xlsx - flickr_output_100.csv"
OUT_FILE = "flickr_clean.csv"

# Tel Aviv bounding box (lon/lat) and a valid Flickr photo-URL pattern
LON_MIN, LON_MAX = 34.70, 34.95
LAT_MIN, LAT_MAX = 31.95, 32.20
FLICKR_URL_RE = re.compile(r"^https?://www\.flickr\.com/photos/[^/]+/\d+/?$")

# --- Load -----------------------------------------------------------
df = pd.read_csv(RAW_FILE, skipinitialspace=True)
raw_count = len(df)

# --- 1. Drop rows with any null value -------------------------------
df = df.dropna()

# --- 2. Date validity -----------------------------------------------
# Flickr launched in 2004; current year is 2026. Keep 2004 inclusive.
df = df[(df["year"] >= 2004) & (df["year"] < 2026)]
df = df[(df["month"] >= 1) & (df["month"] <= 12)]
df = df[(df["day"] >= 1) & (df["day"] <= 31)]
# Build a real datetime; impossible dates (e.g. Feb 30) become NaT and drop.
df["date"] = pd.to_datetime(df[["year", "month", "day"]], errors="coerce")
df = df.dropna(subset=["date"])

# --- 3. Bounding box: keep points inside the Tel Aviv area ----------
df = df[df["lon"].between(LON_MIN, LON_MAX) & df["lat"].between(LAT_MIN, LAT_MAX)]

# --- 4. Land check: drop points in the sea --------------------------
# Spatial join against Natural Earth land polygons (WGS-84).
land = gpd.read_file(geodatasets.get_path("naturalearth.land")).to_crs(epsg=4326)
gdf = gpd.GeoDataFrame(
    df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326"
)
gdf = gpd.sjoin(gdf, land[["geometry"]], how="inner", predicate="within").drop(
    columns="index_right"
)

# --- 5. Remove duplicate photos (same user + same photo URL) --------
gdf = gdf.drop_duplicates(subset=["uid", "flickr_url"])

# --- 6. Remove GPS jitter (same user, spot, and timestamp) ----------
gdf = gdf.drop_duplicates(subset=["uid", "lon", "lat", "created_at"])

# --- 7. Validate Flickr URL format ----------------------------------
gdf = gdf[gdf["flickr_url"].astype(str).str.match(FLICKR_URL_RE)]

# --- Save (drop helper geometry/date columns) -----------------------
clean = gdf.drop(columns=["geometry", "date"])
clean.to_csv(OUT_FILE, index=False)

# --- Summary --------------------------------------------------------
removed = raw_count - len(clean)
print(f"Raw rows:     {raw_count:,}")
print(f"Clean rows:   {len(clean):,}  ({removed:,} removed, {removed / raw_count * 100:.1f}%)")
print(f"Unique users: {clean['uid'].nunique():,}")
print(f"Date range:   {gdf['date'].min().date()} -> {gdf['date'].max().date()}")
print(f"Saved -> {OUT_FILE}")
clean.head()

Raw rows:     63,831
Clean rows:   53,120  (10,711 removed, 16.8%)
Unique users: 2,152
Date range:   2004-01-01 -> 2016-04-27
Saved -> flickr_clean.csv


,lon,lat,flickr_url,day,month,year,created_at,uid,idx
0,34.742310,32.033473,https://www.flickr.com/photos/adrianalobba/102...,12,10,2013,10/12/13 11:37,65324157@N00,1
5,34.744022,32.033858,https://www.flickr.com/photos/134883768@N05/20...,16,4,2015,4/16/15 19:03,134883768@N05,6
6,34.744734,32.033348,https://www.flickr.com/photos/62957207@N03/168...,6,4,2015,4/6/15 10:52,62957207@N03,7
7,34.744452,32.032897,https://www.flickr.com/photos/philram/13763676...,10,4,2014,4/10/14 18:44,112113643@N05,8
8,34.744606,32.033837,https://www.flickr.com/photos/iph4n70m/5865381...,7,5,2011,5/7/11 13:27,41305869@N02,9
